# Dataset de features por género — historial de lectura previo a cada calificación

Este notebook arma el dataset de entrenamiento para el recomendador a partir de `datos/data.db`
(`lectores`, `libros`, `interacciones`), agregando **atributos directos** (autor, género y editorial
del libro, año de edición, género y país del lector) más un **perfil de historial por género**: para
cada calificación y cada género de libro, cuántas veces ese mismo lector calificó libros de ese género
*antes* de esa fecha, y el promedio/mínimo/máximo de esas calificaciones previas. También agrega un
**historial del libro en sí** (sección 6.5): cuántas calificaciones tuvo ese libro y cuál fue su
promedio, antes de la fecha de cada interacción.

Decisiones de diseño acordadas (ver detalle en cada sección):

1. **Alcance**: el historial por género es *por lector* (perfil personal); el historial de libro
   (sección 6.5) es *global* (agregado sobre todos los lectores que calificaron ese libro).
2. **Métricas**: 4 columnas por género — `n` (cantidad), `avg` (promedio), `min`, `max`. El historial
   de libro tiene 2 columnas — `n` y `avg`.
3. **Corte temporal**: estrictamente anterior (`fecha_pasada < fecha_actual`); calificaciones del mismo
   día no cuentan como "previas" entre sí — evita fuga de información dentro del mismo día. Mismo
   criterio para el historial por género y para el historial de libro.
4. **Género de libro**: se normaliza con `LOWER(TRIM(...))` y además con **fuzzy matching** (sección 2)
   para fusionar variantes de tipeo/acentuación que la normalización simple no resuelve. Los libros sin
   género cargado (~60% del catálogo) van a una categoría propia `"desconocido"` con sus propias 4
   columnas — no se descartan.
5. **País del lector**: se corrige el método de extracción (sección 2) para manejar ciudades con guión
   en el nombre propio y direcciones con más de un nivel ("Ciudad - Provincia - País").
6. **Sin historial previo**: `n = 0`, y `avg`/`min`/`max` quedan en `NaN` (no en 0) — XGBoost/LightGBM
   manejan `NaN` nativamente y así no se confunde "nunca calificó este género"/"nadie calificó este
   libro todavía" con "lo calificó mal".
7. **Filas descartadas**: 1 fila con `fecha` corrupta (`"muy poco creíbles"`) y las interacciones cuyo
   `id_libro`/`id_lector` no tiene metadata en `libros`/`lectores` (no se puede enriquecer ni fechar).

Resultado: una fila por interacción válida (lector, libro, fecha, rating), con las columnas base más
las columnas de historial por género (géneros normalizados y fusionados × 4 métricas) más las columnas
de historial de libro (`libro_hist__n`, `libro_hist__avg`). Esta tabla es directamente el set de
entrenamiento para un modelo de regresión de rating (XGBoost/LightGBM); para puntuar candidatos nuevos
después, se reusa el mismo perfil por género/libro tomando el último estado de cada lector/libro en vez
de un corte por fecha.

In [1]:
import re
import sqlite3
import time
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DB_PATH = "datos/data.db"
OUTPUT_PATH = "datos/dataset_features_genero.csv"


## 1. Carga y exploración rápida

Cargamos las tres tablas completas para ver su forma y detectar los problemas de calidad de datos que
vamos a manejar explícitamente más abajo.

In [2]:
conn = sqlite3.connect(DB_PATH)
lectores = pd.read_sql_query("SELECT * FROM lectores", conn)
libros = pd.read_sql_query("SELECT * FROM libros", conn)
interacciones = pd.read_sql_query("SELECT * FROM interacciones", conn)
conn.close()

print("lectores:", lectores.shape)
print("libros:", libros.shape)
print("interacciones:", interacciones.shape)
interacciones.head()

lectores: (11285, 5)
libros: (128743, 9)
interacciones: (461408, 4)


,id_lector,id_libro,fecha,rating
0,bioaqua,las-puertas-de-anubis,12-01-2019,4
1,bioaqua,antonio-b-el-ruso-ciudadano-de-tercera,24-12-2018,10
2,bioaqua,la-momia,24-12-2018,10
3,bioaqua,el-clan-del-oso-cavernario-los-hijos-de-la-tie...,24-12-2018,10
4,bioaqua,dime-quien-soy-1,24-12-2018,10


In [3]:
fecha_ok = interacciones["fecha"].str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
libro_ok = interacciones["id_libro"].isin(set(libros["id_libro"]))
lector_ok = interacciones["id_lector"].isin(set(lectores["id_lector"]))

print(f"fecha con formato inválido ({(~fecha_ok).sum()} filas): {interacciones.loc[~fecha_ok, 'fecha'].tolist()}")
print(f"id_libro sin metadata en libros:   {(~libro_ok).sum()} filas")
print(f"id_lector sin metadata en lectores: {(~lector_ok).sum()} filas")

validas = fecha_ok & libro_ok & lector_ok
print(f"\ninteracciones: {len(interacciones)} -> {validas.sum()} filas válidas "
      f"(se descartan {(~validas).sum()})")

print(f"\nlibros sin género cargado: {(libros['genero'].isna() | (libros['genero'].str.strip() == '')).sum()} "
      f"de {len(libros)} ({(libros['genero'].isna() | (libros['genero'].str.strip() == '')).mean():.0%})")

fecha con formato inválido (1 filas): ['muy poco creíbles']
id_libro sin metadata en libros:   212 filas
id_lector sin metadata en lectores: 122 filas

interacciones: 461408 -> 461073 filas válidas (se descartan 335)

libros sin género cargado: 78222 de 128743 (61%)


## 2. Normalización con fuzzy matching: género y país

`LOWER(TRIM(...))` ya colapsa diferencias de mayúscula/espacios, pero sobreviven variantes de **tipeo**
que no son un problema de mayúsculas: `"biografiás, memorias"` vs `"biografías, memorias"` (la tilde
está en la letra equivocada), `"clasicos de la literatura"` vs `"clásicos de la literatura"` (sin
tilde). Como el género es un feature central del dataset, usamos **similitud de texto** (`rapidfuzz`,
distancia de edición) para detectarlas y fusionarlas — pero con cuidado: fusionar por un umbral de
similitud a ciegas es peligroso (más abajo mostramos un caso real donde haría de cuenta que Australia y
Austria son el mismo país). La regla es: agrupamos por similitud, pero cada fusión se revisa antes de
aplicarse.

Con el país pasa algo distinto: el problema no es de tipeo sino del método de extracción. La expresión
`SUBSTR(vive_en, INSTR(vive_en, '-') + 1)` corta en el **primer** guión, asumiendo el formato
`"Ciudad - País"`. Pero hay ciudades con guión en su propio nombre (`"Vitoria-Gasteiz"`,
`"Rivas-Vaciamadrid"`) y direcciones con más de un nivel (`"Villa La Angostura - Neuquén - Argentina"`),
así que el primer guión no siempre es el separador correcto — el país queda con un fragmento de ciudad
pegado adelante (`"gasteiz - españa"` en vez de `"españa"`). El fix es tomar el **último** guión, no el
primero (SQLite no tiene una función limpia para esto, así que lo resolvemos en pandas).

In [4]:
def agrupar_por_similitud(items, threshold, scorer=fuzz.ratio):
    # une items en clusters (union-find) segun similitud; devuelve tambien los pares que dispararon cada union
    padre = {it: it for it in items}

    def find(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            padre[ra] = rb

    pares = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            score = scorer(items[i], items[j])
            if score >= threshold:
                pares.append((items[i], items[j], score))
                union(items[i], items[j])

    clusters = {}
    for it in items:
        clusters.setdefault(find(it), []).append(it)
    return clusters, pares


In [5]:
### género ###
conn = sqlite3.connect(DB_PATH)
q_conteo_genero = """
SELECT LOWER(TRIM(l.genero)) AS g, COUNT(*) AS n
FROM interacciones i JOIN libros l ON l.id_libro = i.id_libro
WHERE l.genero IS NOT NULL AND TRIM(l.genero) <> ''
GROUP BY g
"""
conteo_genero = pd.read_sql_query(q_conteo_genero, conn).set_index("g")["n"].to_dict()
conn.close()

generos_reales = sorted(conteo_genero.keys())
UMBRAL_GENERO = 80  # validado abajo: separa limpio los tipeos reales de géneros distintos

clusters_genero, pares_genero = agrupar_por_similitud(generos_reales, UMBRAL_GENERO)

print(f"géneros reales (sin 'desconocido'): {len(generos_reales)}")
print(f"pares con similitud >= {UMBRAL_GENERO}:")
for a, b, s in sorted(pares_genero, key=lambda x: -x[2]):
    print(f"  {s:5.1f}  {a!r} ({conteo_genero[a]} calif.) <-> {b!r} ({conteo_genero[b]} calif.)")

# canónico = la variante con más calificaciones (la más confiable / probablemente la bien escrita)
genero_canon = {"desconocido": "desconocido"}
for miembros in clusters_genero.values():
    canonico = max(miembros, key=lambda m: conteo_genero.get(m, 0))
    for m in miembros:
        genero_canon[m] = canonico

fusionados = {k: v for k, v in genero_canon.items() if k != v}
print(f"\n{len(fusionados)} géneros fusionados -> quedan "
      f"{len(set(genero_canon.values()))} categorías canónicas (de {len(generos_reales) + 1})")
for k, v in fusionados.items():
    print(f"  {k!r} -> {v!r}")

géneros reales (sin 'desconocido'): 54
pares con similitud >= 80:
   96.0  'clasicos de la literatura' (15 calif.) <-> 'clásicos de la literatura' (24845 calif.)
   90.0  'biografiás, memorias' (6 calif.) <-> 'biografías, memorias' (10093 calif.)

2 géneros fusionados -> quedan 53 categorías canónicas (de 55)
  'biografiás, memorias' -> 'biografías, memorias'
  'clasicos de la literatura' -> 'clásicos de la literatura'


In [6]:
### país (auditoría) ###
pais_first_hyphen = (
    lectores["vive_en"].str.lower().str.split("-", n=1).str[-1].str.strip()
)
pais_last_hyphen = (
    lectores["vive_en"].str.rsplit("-", n=1).str[-1].str.strip().str.lower()
    .replace({"": np.nan, "¿?": np.nan})
)

afectados = (pais_first_hyphen.str.contains("-", na=False))
print(f"direcciones donde el primer guión da un país 'sucio' (con guión adentro): {afectados.sum()}")

paises = sorted(pais_last_hyphen.dropna().unique())
print(f"\npaíses distintos con el método corregido (último guión): {len(paises)}")

UMBRAL_PAIS = 80
clusters_pais, pares_pais = agrupar_por_similitud(paises, UMBRAL_PAIS)
print(f"\npares de países con similitud >= {UMBRAL_PAIS}:")
for a, b, s in sorted(pares_pais, key=lambda x: -x[2]):
    print(f"  {s:5.1f}  {a!r} <-> {b!r}")

print(
    "\nNINGUNO de estos pares se fusiona: 'australia'/'austria' y "
    "'netherlands antilles'/'netherlands the' tienen alta similitud de texto pero son países/"
    "territorios reales y distintos, no errores de tipeo. Fusionarlos por el puntaje de similitud "
    "sería un error de datos -- acá el fuzzy matching se usa para CONFIRMAR que no hace falta tocar "
    "nada más allá de arreglar el método de extracción (último guión) y los placeholders ('', '¿?')."
)

direcciones donde el primer guión da un país 'sucio' (con guión adentro): 42

países distintos con el método corregido (último guión): 61

pares de países con similitud >= 80:
   87.5  'australia' <-> 'austria'
   80.0  'netherlands antilles' <-> 'netherlands the'

NINGUNO de estos pares se fusiona: 'australia'/'austria' y 'netherlands antilles'/'netherlands the' tienen alta similitud de texto pero son países/territorios reales y distintos, no errores de tipeo. Fusionarlos por el puntaje de similitud sería un error de datos -- acá el fuzzy matching se usa para CONFIRMAR que no hace falta tocar nada más allá de arreglar el método de extracción (último guión) y los placeholders ('', '¿?').


## 3. Columnas base por SQL (atributos directos)

Join `interacciones ⋈ libros ⋈ lectores` con `INNER JOIN`, que ya descarta automáticamente las
interacciones huérfanas (sin metadata de libro o lector). El filtro `GLOB` descarta la única fila con
fecha corrupta. `genero_libro` y `pais_lector` se resuelven después en pandas con el mapeo canónico de
género y la extracción de país corregida de la sección 2.

In [7]:
query_base = """
SELECT
    i.id_lector,
    i.id_libro,
    i.fecha,
    i.rating,
    l.autor        AS autor_libro,
    l.genero        AS genero_libro_raw,
    l.editorial     AS editorial_libro,
    l.anio_edicion  AS anio_edicion_libro,
    r.genero        AS genero_lector,
    r.vive_en       AS vive_en_lector
FROM interacciones i
JOIN libros l    ON l.id_libro = i.id_libro
JOIN lectores r  ON r.id_lector = i.id_lector
WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
"""

conn = sqlite3.connect(DB_PATH)
dataset = pd.read_sql_query(query_base, conn)
conn.close()

dataset["fecha_iso"] = pd.to_datetime(dataset["fecha"], format="%d-%m-%Y")
dataset["rating"] = dataset["rating"].astype(float)
dataset["genero_lector"] = dataset["genero_lector"].replace({"-": np.nan, "": np.nan})

# anio_edicion_libro llega con basura de texto (' (200', ' crít', etc.) -> a numerico, y
# los pocos valores fuera de un rango plausible (ej. '201') tambien quedan en NaN
anio_num = pd.to_numeric(dataset["anio_edicion_libro"], errors="coerce")
dataset["anio_edicion_libro"] = anio_num.where(anio_num.between(1400, 2026))

dataset["genero_libro"] = (
    dataset["genero_libro_raw"].str.strip().str.lower().map(genero_canon).fillna("desconocido")
)
dataset["pais_lector"] = (
    dataset["vive_en_lector"].str.rsplit("-", n=1).str[-1].str.strip().str.lower()
    .replace({"": np.nan, "¿?": np.nan})
)
dataset = dataset.drop(columns=["genero_libro_raw", "vive_en_lector"])

print(dataset.shape)
dataset.head()

(461073, 11)


,id_lector,id_libro,fecha,rating,autor_libro,editorial_libro,anio_edicion_libro,genero_lector,fecha_iso,genero_libro,pais_lector
0,bioaqua,las-puertas-de-anubis,12-01-2019,4.0,"POWERS, TIM",GIGAMESH,2015.0,Hombre,2019-01-12,"fantástica, ciencia ficción",españa
1,bioaqua,antonio-b-el-ruso-ciudadano-de-tercera,24-12-2018,10.0,"PINILLA, RAMIRO",TUSQUETS,2007.0,Hombre,2018-12-24,no ficción,españa
2,bioaqua,la-momia,24-12-2018,10.0,"RICE, ANNE",BYBLOS,2005.0,Hombre,2018-12-24,histórica y aventuras,españa
3,bioaqua,el-clan-del-oso-cavernario-los-hijos-de-la-tie...,24-12-2018,10.0,"AUEL, JEAN MARIE",MAEVA,2024.0,Hombre,2018-12-24,histórica y aventuras,españa
4,bioaqua,dime-quien-soy-1,24-12-2018,10.0,"NAVARRO, JULIA",PLAZA & JANÉS,2010.0,Hombre,2018-12-24,narrativa,españa


## 4. Historial acumulado por lector × género (SQL, leak-safe)

Necesitamos, para cada interacción y cada género canónico, el `n`/`avg`/`min`/`max` de las
calificaciones que **ese lector** hizo en libros de **ese género** antes de esa fecha.

Escribirlo como un único `SELECT` con una subquery correlacionada por cada columna no escala en
SQLite. En cambio, construimos un **"changelog"** compacto con *window functions*: una fila por cada
(lector, género, fecha) en la que ese lector calificó algo de ese género, con el acumulado hasta e
incluyendo esa fecha.

El mapeo canónico de género (sección 2) se aplica **adentro** de la subquery `base`, antes de agregar
por día y antes de la ventana acumulada — si lo aplicáramos después de calcular los acumulados por
separado para cada variante de tipeo, el conteo/promedio de cada una quedaría incompleto en vez de ser
la fusión real de ambas. Como es un mapeo calculado en Python (por el fuzzy matching), lo pasamos a SQL
con una tabla temporal.

In [8]:
conn = sqlite3.connect(DB_PATH)
conn.execute("CREATE TEMP TABLE genero_canon (genero_norm TEXT PRIMARY KEY, genero_canonico TEXT NOT NULL)")
conn.executemany("INSERT INTO genero_canon VALUES (?, ?)", list(genero_canon.items()))

query_changelog = """
WITH base AS (
    SELECT
        i.id_lector,
        COALESCE(gc.genero_canonico, 'desconocido') AS genero_norm,
        substr(i.fecha, 7, 4) || '-' || substr(i.fecha, 4, 2) || '-' || substr(i.fecha, 1, 2) AS fecha_iso,
        i.rating AS rating
    FROM interacciones i
    JOIN libros l    ON l.id_libro = i.id_libro
    JOIN lectores r  ON r.id_lector = i.id_lector
    LEFT JOIN genero_canon gc ON gc.genero_norm = LOWER(TRIM(l.genero))
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
),
por_dia AS (
    -- si un lector calificó >1 libro del mismo género el mismo día, van al mismo punto
    -- del changelog: el corte de "historial previo" es estrictamente anterior a la fecha
    SELECT
        id_lector, genero_norm, fecha_iso,
        COUNT(*)    AS n_dia,
        SUM(rating) AS suma_dia,
        MIN(rating) AS min_dia,
        MAX(rating) AS max_dia
    FROM base
    GROUP BY id_lector, genero_norm, fecha_iso
)
SELECT
    id_lector, genero_norm, fecha_iso,
    SUM(n_dia)    OVER w AS n_acum,
    SUM(suma_dia) OVER w AS suma_acum,
    MIN(min_dia)  OVER w AS min_acum,
    MAX(max_dia)  OVER w AS max_acum
FROM por_dia
WINDOW w AS (
    PARTITION BY id_lector, genero_norm
    ORDER BY fecha_iso
    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY id_lector, genero_norm, fecha_iso
"""

t0 = time.time()
changelog = pd.read_sql_query(query_changelog, conn)
conn.close()
changelog["fecha_iso"] = pd.to_datetime(changelog["fecha_iso"])

print(f"changelog: {changelog.shape} en {time.time() - t0:.1f}s")
print(f"géneros canónicos distintos (incluye 'desconocido'): {changelog['genero_norm'].nunique()}")
changelog.head()

changelog: (246671, 7) en 1.5s
géneros canónicos distintos (incluye 'desconocido'): 53


,id_lector,genero_norm,fecha_iso,n_acum,suma_acum,min_acum,max_acum
0,-2,clásicos de la literatura,2016-01-24,2,19,9,10
1,-2,histórica y aventuras,2017-07-06,1,6,6,6
2,-2,infantil y juvenil,2016-01-24,2,16,7,9
3,-2,narrativa,2016-01-24,1,10,10,10
4,-2,"novela negra, intriga, terror",2016-01-24,3,22,6,9


## 5. De changelog a features por fila (pivot + as-of merge)

El changelog es "largo": una fila por (lector, género, fecha de cambio). Lo pasamos a "ancho"
(una fila por lector×fecha, una columna por género×métrica) y propagamos hacia adelante (`ffill`)
los géneros que ese lector no tocó en esa fecha puntual, para que cada fila represente el estado
completo del perfil del lector en ese momento.

Después, para cada interacción buscamos el estado del perfil **inmediatamente anterior** a su fecha
con `pd.merge_asof(..., direction="backward", allow_exact_matches=False)` — un *as-of join*, que
SQLite no soporta nativamente y que sí es el tipo de operación para el que pandas está pensado.
`allow_exact_matches=False` es lo que garantiza el corte "estrictamente anterior".

In [9]:
def slugify(genero_norm: str) -> str:
    s = unicodedata.normalize("NFKD", genero_norm).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


genero_slug = {g: slugify(g) for g in changelog["genero_norm"].unique()}
assert len(set(genero_slug.values())) == len(genero_slug), (
    "colisión de slugs entre géneros distintos -- revisar el fuzzy matching de la sección 2"
)

wide_hist = changelog.pivot(
    index=["id_lector", "fecha_iso"], columns="genero_norm",
    values=["n_acum", "suma_acum", "min_acum", "max_acum"],
)
wide_hist.columns = [
    f"genero_hist__{metric.replace('_acum', '')}__{genero_slug[genero]}"
    for metric, genero in wide_hist.columns
]
wide_hist = wide_hist.reset_index().sort_values("fecha_iso")

n_cols = [c for c in wide_hist.columns if c.startswith("genero_hist__n__")]
otras_cols = [c for c in wide_hist.columns if c.startswith("genero_hist__") and c not in n_cols]

wide_hist[n_cols] = wide_hist.groupby("id_lector")[n_cols].ffill()
wide_hist[otras_cols] = wide_hist.groupby("id_lector")[otras_cols].ffill()

print(f"perfil ancho: {wide_hist.shape} (snapshots de perfil por lector×fecha de cambio)")

perfil ancho: (156877, 214) (snapshots de perfil por lector×fecha de cambio)


In [10]:
dataset_sorted = dataset.sort_values("fecha_iso").reset_index(drop=True)
wide_hist_sorted = wide_hist.sort_values("fecha_iso").reset_index(drop=True)

t0 = time.time()
dataset_full = pd.merge_asof(
    dataset_sorted, wide_hist_sorted,
    on="fecha_iso", by="id_lector",
    direction="backward", allow_exact_matches=False,  # estrictamente anterior a la fecha
)
print(f"merge_asof: {dataset_full.shape} en {time.time() - t0:.1f}s")

n_cols = [c for c in dataset_full.columns if c.startswith("genero_hist__n__")]
suma_cols = [c for c in dataset_full.columns if c.startswith("genero_hist__suma__")]

# sin historial previo -> n=0 (no hubo ninguna calificación previa en ese género)
dataset_full[n_cols] = dataset_full[n_cols].fillna(0).astype("int32")

# avg = suma / n; si n=0 queda NaN (no 0) para no confundir "sin historial" con "calificó mal"
avg_cols = {}
for suma_c in suma_cols:
    genero_tag = suma_c[len("genero_hist__suma__"):]
    n_c = f"genero_hist__n__{genero_tag}"
    avg_c = f"genero_hist__avg__{genero_tag}"
    avg_cols[avg_c] = (dataset_full[suma_c] / dataset_full[n_c].replace(0, np.nan)).astype("float32")

dataset_full = pd.concat([dataset_full.drop(columns=suma_cols), pd.DataFrame(avg_cols)], axis=1)

min_max_cols = [c for c in dataset_full.columns
                if c.startswith("genero_hist__min__") or c.startswith("genero_hist__max__")]
dataset_full[min_max_cols] = dataset_full[min_max_cols].astype("float32")

hist_cols = sorted(c for c in dataset_full.columns if c.startswith("genero_hist__"))
base_cols = [c for c in dataset_full.columns if c not in hist_cols]
dataset_full = dataset_full[base_cols + hist_cols]

print(f"columnas de historial por género: {len(hist_cols)} ({len(hist_cols) // 4} géneros x 4 métricas)")
print(f"dataset final: {dataset_full.shape}")
dataset_full.head()

merge_asof: (461073, 223) en 0.3s
columnas de historial por género: 212 (53 géneros x 4 métricas)
dataset final: (461073, 223)


,id_lector,id_libro,fecha,rating,autor_libro,editorial_libro,anio_edicion_libro,genero_lector,fecha_iso,genero_libro,pais_lector,genero_hist__avg__arte,genero_hist__avg__autoayuda_y_espiritualidad,genero_hist__avg__biografias_memorias,genero_hist__avg__clasicos_de_la_literatura,genero_hist__avg__cocina,genero_hist__avg__comics_novela_grafica,genero_hist__avg__deportes_y_juegos,genero_hist__avg__derecho,genero_hist__avg__desconocido,...,genero_hist__n__literatura_contemporanea,genero_hist__n__marketing_y_publicidad,genero_hist__n__medicina,genero_hist__n__medicina_divulgativa,genero_hist__n__musica,genero_hist__n__narrativa,genero_hist__n__naturaleza_y_ciencia,genero_hist__n__no_ficcion,genero_hist__n__novela,genero_hist__n__novela_negra,genero_hist__n__novela_negra_intriga_terror,genero_hist__n__peliculas,genero_hist__n__poesia,genero_hist__n__poesia_teatro,genero_hist__n__psicologia_y_pedagogia,genero_hist__n__romantica_erotica,genero_hist__n__television,genero_hist__n__varios,genero_hist__n__varios_otros_generos,genero_hist__n__viajes_en_bolsillo
0,jc01,en-el-blanco,24-02-2008,4.0,"FOLLETT, KEN",DEBOLSILLO,2006.0,Hombre,2008-02-24,ficción literaria,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,jc01,vuelo-final,24-02-2008,4.0,"FOLLETT, KEN",DEBOLSILLO,2003.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,jc01,la-clave-esta-en-rebeca,24-02-2008,6.0,"FOLLETT, KEN",DEBOLSILLO,2003.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,jc01,un-mundo-sin-fin-los-pilares-de-la-tierra-2,24-02-2008,6.0,"FOLLETT, KEN",PLAZA & JANÉS,2007.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,jc01,el-gran-gatsby,24-02-2008,8.0,"SCOTT FITZGERALD, FRANCIS",ALFAGUARA,2019.0,Hombre,2008-02-24,narrativa,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 6. Validación: chequeo manual anti-fuga

Para un lector con historial largo, recalculamos a mano (con pandas, filtrando por fecha) el
`n`/`avg`/`min`/`max` de su género más frecuente *antes* de su última calificación, y lo comparamos
contra lo que salió del pipeline. Deben coincidir exactamente. `dataset["genero_libro"]` ya está en
su forma canónica (post fuzzy matching), así que no hace falta volver a normalizarlo acá.

In [11]:
lector_muestra = (
    interacciones[validas].groupby("id_lector").size().sort_values(ascending=False).index[3]
)

hist_lector = dataset_full.loc[dataset_full["id_lector"] == lector_muestra, "fecha_iso"]
fila = dataset_full[
    (dataset_full["id_lector"] == lector_muestra) & (dataset_full["fecha_iso"] == hist_lector.max())
].iloc[0]

genero_objetivo = fila["genero_libro"]
slug_objetivo = slugify(genero_objetivo)

previas = dataset[
    (dataset["id_lector"] == lector_muestra)
    & (dataset["genero_libro"] == genero_objetivo)
    & (dataset["fecha_iso"] < fila["fecha_iso"])
]

print(f"lector: {lector_muestra} | género: {genero_objetivo!r} -> slug: {slug_objetivo}")
print(f"manual  -> n={len(previas)}, avg={previas['rating'].mean()}, "
      f"min={previas['rating'].min()}, max={previas['rating'].max()}")
print(f"pipeline -> n={fila[f'genero_hist__n__{slug_objetivo}']}, "
      f"avg={fila[f'genero_hist__avg__{slug_objetivo}']}, "
      f"min={fila[f'genero_hist__min__{slug_objetivo}']}, "
      f"max={fila[f'genero_hist__max__{slug_objetivo}']}")

assert len(previas) == fila[f"genero_hist__n__{slug_objetivo}"], "no coincide el conteo -> hay fuga o error"
print("\nOK: el pipeline coincide con el cálculo manual (sin fuga de información).")

lector: afraile | género: 'histórica y aventuras' -> slug: historica_y_aventuras
manual  -> n=349, avg=6.836676217765043, min=4.0, max=10.0
pipeline -> n=349, avg=6.836676120758057, min=4.0, max=10.0

OK: el pipeline coincide con el cálculo manual (sin fuga de información).


In [12]:
# caso sin historial previo: primera calificación de cada lector en un género nunca antes calificado
sin_historial = dataset_full[dataset_full["genero_hist__n__narrativa"] == 0]
print(f"filas sin historial previo en 'narrativa': {len(sin_historial)}")
print(sin_historial[["genero_hist__n__narrativa", "genero_hist__avg__narrativa",
                      "genero_hist__min__narrativa", "genero_hist__max__narrativa"]].head())
assert sin_historial["genero_hist__avg__narrativa"].isna().all(), "avg debería ser NaN, no 0, sin historial"
print("OK: avg/min/max quedan en NaN (no en 0) cuando no hay historial previo.")

filas sin historial previo en 'narrativa': 160300
   genero_hist__n__narrativa  genero_hist__avg__narrativa  genero_hist__min__narrativa  genero_hist__max__narrativa
0                          0                          NaN                          NaN                          NaN
1                          0                          NaN                          NaN                          NaN
2                          0                          NaN                          NaN                          NaN
3                          0                          NaN                          NaN                          NaN
4                          0                          NaN                          NaN                          NaN
OK: avg/min/max quedan en NaN (no en 0) cuando no hay historial previo.


## 6.5 Historial del libro (global): evaluaciones previas y promedio (leak-safe)

Además del perfil por género de cada lector, agregamos dos columnas con el historial **del libro en
sí**, agregado sobre *todos* los lectores que lo calificaron (no solo el lector de esa fila):
`libro_hist__n` (cuántas calificaciones tuvo ese libro antes de la fecha de esta interacción) y
`libro_hist__avg` (el promedio de esas calificaciones previas).

Es la misma lógica leak-safe de las secciones 4-5 (changelog acumulado por día + `merge_asof` con
`direction="backward", allow_exact_matches=False`), pero la partición del acumulado es por `id_libro`
en vez de por `id_lector` × género: no importa qué lector calificó antes, solo que haya sido
*estrictamente antes* de esta fila — con el mismo corte a nivel día (calificaciones del mismo libro el
mismo día, de cualquier lector, no cuentan como "previas" entre sí, evitando fuga de información dentro
del mismo día). Sin historial previo, `n = 0` y `avg` queda en `NaN`, mismo criterio que en la sección
4 (para no confundir "nadie lo calificó todavía" con "lo calificaron mal").

In [13]:
query_changelog_libro = """
WITH base AS (
    SELECT
        i.id_libro,
        substr(i.fecha, 7, 4) || '-' || substr(i.fecha, 4, 2) || '-' || substr(i.fecha, 1, 2) AS fecha_iso,
        i.rating AS rating
    FROM interacciones i
    JOIN libros l    ON l.id_libro = i.id_libro
    JOIN lectores r  ON r.id_lector = i.id_lector
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
),
por_dia AS (
    -- si el libro recibió >1 calificación el mismo día (de cualquier lector), van al mismo punto
    -- del changelog: el corte de "historial previo" es estrictamente anterior a la fecha
    SELECT
        id_libro, fecha_iso,
        COUNT(*)    AS n_dia,
        SUM(rating) AS suma_dia
    FROM base
    GROUP BY id_libro, fecha_iso
)
SELECT
    id_libro, fecha_iso,
    SUM(n_dia)    OVER w AS n_acum,
    SUM(suma_dia) OVER w AS suma_acum
FROM por_dia
WINDOW w AS (
    PARTITION BY id_libro
    ORDER BY fecha_iso
    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY id_libro, fecha_iso
"""

conn = sqlite3.connect(DB_PATH)
t0 = time.time()
changelog_libro = pd.read_sql_query(query_changelog_libro, conn)
conn.close()
changelog_libro["fecha_iso"] = pd.to_datetime(changelog_libro["fecha_iso"])

print(f"changelog libro: {changelog_libro.shape} en {time.time() - t0:.1f}s")
changelog_libro.head()

changelog libro: (443403, 4) en 1.2s


,id_libro,fecha_iso,n_acum,suma_acum
0,-incluso-el-olvido,2016-07-08,1,6
1,1-or-w-1,2017-05-29,1,6
2,10-000-anos-mirando-estrellas,2014-03-26,1,7
3,10-anos-con-mafalda,2008-06-04,1,8
4,10-anos-con-mafalda,2008-06-28,2,18


In [14]:
dataset_full_sorted = dataset_full.sort_values("fecha_iso").reset_index(drop=True)
changelog_libro_sorted = changelog_libro.sort_values("fecha_iso").reset_index(drop=True)

t0 = time.time()
dataset_full = pd.merge_asof(
    dataset_full_sorted, changelog_libro_sorted,
    on="fecha_iso", by="id_libro",
    direction="backward", allow_exact_matches=False,  # estrictamente anterior a la fecha
)
print(f"merge_asof libro: {dataset_full.shape} en {time.time() - t0:.1f}s")

# sin historial previo -> n=0; avg queda en NaN (no en 0), mismo criterio que en la sección 5
libro_hist_cols = pd.DataFrame({
    "libro_hist__n": dataset_full["n_acum"].fillna(0).astype("int32"),
    "libro_hist__avg": (dataset_full["suma_acum"] / dataset_full["n_acum"]).astype("float32"),
})
dataset_full = pd.concat([dataset_full.drop(columns=["n_acum", "suma_acum"]), libro_hist_cols], axis=1)

print(f"dataset final: {dataset_full.shape}")
dataset_full[["id_lector", "id_libro", "fecha", "rating", "libro_hist__n", "libro_hist__avg"]].head()

merge_asof libro: (461073, 225) en 0.1s
dataset final: (461073, 225)


,id_lector,id_libro,fecha,rating,libro_hist__n,libro_hist__avg
0,jc01,en-el-blanco,24-02-2008,4.0,0,NaN
1,jc01,vuelo-final,24-02-2008,4.0,0,NaN
2,jc01,la-clave-esta-en-rebeca,24-02-2008,6.0,0,NaN
3,jc01,un-mundo-sin-fin-los-pilares-de-la-tierra-2,24-02-2008,6.0,0,NaN
4,jc01,el-gran-gatsby,24-02-2008,8.0,0,NaN


In [15]:
# validación manual anti-fuga, igual que en la sección 6 pero a nivel libro (no lector x género)
libro_muestra = (
    interacciones[validas].groupby("id_libro").size().sort_values(ascending=False).index[0]
)

hist_libro = dataset_full.loc[dataset_full["id_libro"] == libro_muestra, "fecha_iso"]
fila = dataset_full[
    (dataset_full["id_libro"] == libro_muestra) & (dataset_full["fecha_iso"] == hist_libro.max())
].iloc[0]

previas = dataset[
    (dataset["id_libro"] == libro_muestra) & (dataset["fecha_iso"] < fila["fecha_iso"])
]

print(f"libro: {libro_muestra!r}")
print(f"manual   -> n={len(previas)}, avg={previas['rating'].mean()}")
print(f"pipeline -> n={fila['libro_hist__n']}, avg={fila['libro_hist__avg']}")

assert len(previas) == fila["libro_hist__n"], "no coincide el conteo -> hay fuga o error"
print("\nOK: el historial de libro coincide con el cálculo manual (sin fuga de información).")

libro: 'la-sombra-del-viento-el-cementerio-de-los-libros-olvidados-1'
manual   -> n=2227, avg=7.917826672653795
pipeline -> n=2227, avg=7.9178266525268555

OK: el historial de libro coincide con el cálculo manual (sin fuga de información).


## 7. Guardado

Guardamos el dataset completo (columnas base + historial por género) a CSV. También guardamos la
tabla `genero_slug` (género canónico -> sufijo de columna) para poder mapear las columnas de vuelta
a su nombre legible, y el mapeo `genero_canon` completo (género original -> canónico) como registro
de qué se fusionó.

In [16]:
dataset_full.to_csv(OUTPUT_PATH, index=False)
print(f"guardado en {OUTPUT_PATH} ({len(dataset_full):,} filas, {dataset_full.shape[1]} columnas)")

pd.Series(genero_slug, name="slug").rename_axis("genero_canonico").reset_index().to_csv(
    "datos/genero_slug_lookup.csv", index=False
)
pd.Series(genero_canon, name="genero_canonico").rename_axis("genero_original").reset_index().to_csv(
    "datos/genero_canon_lookup.csv", index=False
)
print(f"lookup de géneros guardado en datos/genero_slug_lookup.csv ({len(genero_slug)} géneros canónicos)")
print(f"mapeo original->canónico guardado en datos/genero_canon_lookup.csv ({len(genero_canon)} entradas)")

guardado en datos/dataset_features_genero.csv (461,073 filas, 225 columnas)
lookup de géneros guardado en datos/genero_slug_lookup.csv (53 géneros canónicos)
mapeo original->canónico guardado en datos/genero_canon_lookup.csv (55 entradas)


## 8. Resumen

- Dataset final: una fila por interacción válida, con columnas base (`autor_libro`, `genero_libro`,
  `editorial_libro`, `anio_edicion_libro`, `genero_lector`, `pais_lector`), columnas de historial por
  género (géneros canónicos × `n`/`avg`/`min`/`max`) y columnas de historial de libro (`libro_hist__n`,
  `libro_hist__avg`), todas calculadas *antes* de la fecha de cada calificación — sin fuga de
  información hacia el futuro (validado en las secciones 6 y 6.5).
- **Género**: `LOWER(TRIM(...))` + fuzzy matching (`rapidfuzz`, umbral 80 sobre `fuzz.ratio`) fusionó 2
  pares de variantes de tipeo/tilde en 2 categorías canónicas, elegidas por la variante con más
  calificaciones. El resto de pares con similitud alta (géneros de distinta granularidad, ej.
  `"novela negra"` vs `"novela negra, intriga, terror"`) quedaron muy por debajo del umbral y se
  dejaron separados a propósito — no son errores de tipeo, son categorías distintas.
- **País**: se corrigió la extracción para tomar el segmento después del **último** guión (no el
  primero), por ciudades con guión en su propio nombre y direcciones de más de un nivel. La auditoría
  fuzzy sobre los países resultantes no encontró duplicados reales — los únicos candidatos
  (`"australia"`/`"austria"`, `"netherlands antilles"`/`"netherlands the"`) son países/territorios
  distintos y se dejaron sin fusionar a propósito.
- **Historial de libro**: a diferencia del historial por género (personal, por lector), `libro_hist__n`
  / `libro_hist__avg` son un agregado *global* del libro — cuántas calificaciones tuvo y su promedio,
  contando a todos los lectores, antes de la fecha de la fila. Es una señal de popularidad/calidad
  percibida del libro hasta ese momento, útil sobre todo para libros nuevos o poco calificados donde el
  historial por género del lector puede ser escaso.
- **Siguiente paso** (fuera de este notebook): entrenar XGBoost/LightGBM sobre `rating` con estas
  columnas, y para los lectores de `datos/ejemplo.csv` puntuar todos los libros que cada uno no leyó,
  usando el *último* estado del perfil de cada lector/libro (en vez de un corte por fecha) para las
  columnas de historial por género y de historial de libro.